# From Textbook Clusters to Production Lakehouses
**JHU EN.601.412/612 Big Data Systems, guest lecture demo (Part 1), open source version**

Runs on a laptop with open source Spark + Delta Lake + DuckDB. No Databricks needed.

**Prerequisites**
* Java 17 (`java -version`), with `JAVA_HOME` set
* Python 3.10 or 3.11 (3.12+ breaks PySpark 3.5), then: `pip install "pyspark==3.5.*" "delta-spark==3.2.*" duckdb pandas jupyterlab setuptools`
  (pyspark and delta-spark versions must match: Spark 3.5 goes with Delta 3.x, Spark 4.0 with Delta 4.0)
* About 8 GB free RAM and 5 GB free disk. If your machine is small, lower `N_EVENTS` in the config cell.

The first run downloads the Delta JARs from Maven, so do it once on a good connection.
Spark UI: http://localhost:4040 while the session is alive.

| Section | What it shows |
|---|---|
| 1.1 | Delta log internals, file-level min/max stats, data skipping before vs after OPTIMIZE ZORDER, copy-on-write vs deletion vectors, time travel |
| 1.2 | Skewed sort-merge join in the Spark UI, then two real join fixes (AQE skew join, broadcast), then the same SQL in DuckDB on the same machine |

In [ ]:
# [BEFORE CLASS] Config
import os, json, time, shutil
import pandas as pd
from pyspark.sql import SparkSession, DataFrame, functions as F
from delta import configure_spark_with_delta_pip
from IPython.display import display as _ipy_display

builder = (SparkSession.builder
    .appName("jhu-lakehouse-demo")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.ui.port", "4040")
    .config("spark.sql.debug.maxToStringFields", "1000"))   # avoids a harmless "Truncated the string representation of a plan" warning
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")   # hide routine Spark WARN messages in cell output
print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

def display(obj, n=20):
    """Stand-in for Databricks display(): Spark DataFrames become pandas tables."""
    if isinstance(obj, DataFrame):
        obj = obj.limit(n).toPandas()
    _ipy_display(obj)

LOCAL = os.path.abspath("jhu_demo_data")   # path for Python and DuckDB
BASE = LOCAL                               # Spark sees the same local path

N_EVENTS = 20_000_000          # lower to 5_000_000 on a small laptop
N_USERS = 1_000_000
HOT_USER_SHARE = 0.40             # 40% of all events belong to user_id = 1 (the skewed key)

spark.conf.set("spark.sql.session.timeZone", "UTC")
# Keep Delta from tidying files for us (these are off by default in open source, set for clarity). We want the "before" state to be messy.
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "false")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "false")
# Start with deletion vectors OFF so we can show copy-on-write first.
spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false")
# Smaller OPTIMIZE target so the compacted table still has enough files to skip.
spark.conf.set("spark.databricks.delta.optimize.maxFileSize", str(16 * 1024 * 1024))

TIMINGS = {}

def timed(label, sql):
    """Run a query to completion without collecting results, record wall time,
    and tag the job so it is easy to find in the Spark UI Jobs/SQL tabs."""
    spark.sparkContext.setJobDescription(label)
    t0 = time.time()
    spark.sql(sql).write.format("noop").mode("overwrite").save()
    TIMINGS[label] = round(time.time() - t0, 2)
    spark.sparkContext.setJobDescription(None)
    print(f"{label}: {TIMINGS[label]}s")

def delta(path):
    return f"delta.`{BASE}/{path}`"

In [ ]:
# [BEFORE CLASS] Generate synthetic data
shutil.rmtree(LOCAL, ignore_errors=True)

COUNTRIES = F.array(*[F.lit(c) for c in ["US", "IN", "BR", "DE", "JP", "GB", "SG", "AU"]])
DEVICES = F.array(*[F.lit(d) for d in ["ios", "android", "web"]])
TIERS = F.array(*[F.lit(t) for t in ["free", "plus", "pro", "enterprise"]])

users = (
    spark.range(1, N_USERS + 1).withColumnRenamed("id", "user_id")
    .withColumn("tier", F.element_at(TIERS, (F.col("user_id") % 4 + 1).cast("int")))
    .withColumn("signup_country", F.element_at(COUNTRIES, (F.col("user_id") % 8 + 1).cast("int")))
)

events = (
    spark.range(N_EVENTS).withColumnRenamed("id", "event_id")
    .withColumn("user_id",
        F.when(F.rand(42) < HOT_USER_SHARE, F.lit(1))
         .otherwise((F.rand(7) * N_USERS).cast("long") + 1))
    # 2026-01-01 UTC plus a random point in the next 90 days
    .withColumn("event_ts", F.timestamp_seconds(F.lit(1767225600) + (F.rand(11) * 90 * 86400).cast("long")))
    .withColumn("country", F.element_at(COUNTRIES, (F.rand(13) * 8).cast("int") + 1))
    .withColumn("device", F.element_at(DEVICES, (F.rand(17) * 3).cast("int") + 1))
    .withColumn("amount", F.round(F.rand(19) * 200, 2))
)

# "Raw lake": 1,000 small Parquet files, rows randomly scattered (the small-file problem)
events.repartition(1000).write.mode("overwrite").parquet(f"{BASE}/raw_events_parquet")

# Unoptimized Delta table: 400 files, every file spans the whole quarter
events.repartition(400).write.format("delta").mode("overwrite").save(f"{BASE}/events_delta")

users.write.format("delta").mode("overwrite").save(f"{BASE}/users_delta")
print("done")

## 1.1 Beyond plain Parquet: what a table format actually adds
Plain Parquet folder = a pile of files. No transactions, no stats index, and the engine has to LIST the directory to find out what exists.

In [ ]:
# The raw lake: just files
raw_files = [f for f in os.listdir(f"{LOCAL}/raw_events_parquet") if f.endswith(".parquet")]
print(f"{len(raw_files)} parquet files, no log, no table-level stats")

In [ ]:
# The Delta table: the transaction log
log_dir = f"{LOCAL}/events_delta/_delta_log"
print(sorted(os.listdir(log_dir)))

with open(f"{log_dir}/00000000000000000000.json") as f:
    actions = [json.loads(line) for line in f]

print("Action types in commit 0:", sorted({k for a in actions for k in a}))
adds = [a["add"] for a in actions if "add" in a]
print(f"{len(adds)} data files registered in commit 0\n")
print("Stats stored for ONE file:")
print(json.dumps(json.loads(adds[0]["stats"]), indent=2))

In [ ]:
# Per-file min/max on event_ts: every file covers the whole quarter
def file_ranges(add_actions, n=10):
    rows = []
    for a in add_actions[:n]:
        s = json.loads(a["stats"])
        rows.append({"file": a["path"][-24:], "rows": s["numRecords"],
                     "min_event_ts": s["minValues"]["event_ts"], "max_event_ts": s["maxValues"]["event_ts"]})
    return pd.DataFrame(rows)

display(file_ranges(adds))

In [ ]:
# Data skipping, done by hand: which files could contain the hour?
LO, HI = pd.Timestamp("2026-02-14 09:00:00", tz="UTC"), pd.Timestamp("2026-02-14 10:00:00", tz="UTC")

def _utc(x):
    t = pd.Timestamp(x)
    return t.tz_localize("UTC") if t.tzinfo is None else t.tz_convert("UTC")

def skip_report(add_actions):
    """Same check the engine runs at planning time: compare each file's
    min/max stats to the WHERE clause and drop files that can't match."""
    must_read = sum(
        1 for a in add_actions
        if _utc(json.loads(a["stats"])["minValues"]["event_ts"]) < HI
        and _utc(json.loads(a["stats"])["maxValues"]["event_ts"]) >= LO)
    total = len(add_actions)
    print(f"{total} files in the table | must read {must_read} | can skip {total - must_read}")

skip_report(adds)

In [ ]:
# Selective query BEFORE optimization  (Spark UI > SQL / DataFrame > scan node: "number of files read")
Q_FILTER = f"""
SELECT country, count(*) AS n, round(sum(amount), 2) AS revenue
FROM {delta('events_delta')}
WHERE event_ts >= '2026-02-14 09:00:00' AND event_ts < '2026-02-14 10:00:00'
GROUP BY country
"""
timed("1.1 filter, unoptimized", Q_FILTER)
display(spark.sql(f"DESCRIBE DETAIL {delta('events_delta')}").select("numFiles", "sizeInBytes"))

In [ ]:
# Compact + cluster the data on the filter column
display(spark.sql(f"OPTIMIZE {delta('events_delta')} ZORDER BY (event_ts)"))
display(spark.sql(f"DESCRIBE DETAIL {delta('events_delta')}").select("numFiles", "sizeInBytes"))

In [ ]:
# Per-file ranges AFTER: non-overlapping, so the engine can skip
latest = sorted(f for f in os.listdir(log_dir) if f.endswith(".json"))[-1]
with open(f"{log_dir}/{latest}") as f:
    new_adds = [json.loads(l)["add"] for l in f if '"add"' in l]
display(file_ranges(sorted(new_adds, key=lambda a: json.loads(a["stats"])["minValues"]["event_ts"]), n=12))

In [ ]:
# Same query AFTER  (compare "number of files read" in the Spark UI)
skip_report(new_adds)
timed("1.1 filter, OPTIMIZE ZORDER", Q_FILTER)

### Row-level changes: copy-on-write vs deletion vectors (merge-on-read)

In [ ]:
# DELETE #1, deletion vectors OFF: whole files get rewritten
spark.sql(f"DELETE FROM {delta('events_delta')} WHERE country = 'BR' AND amount < 1")

In [ ]:
# DELETE #2, deletion vectors ON: files stay, a small bitmap marks dead rows
spark.sql(f"ALTER TABLE {delta('events_delta')} SET TBLPROPERTIES ('delta.enableDeletionVectors' = true)")
spark.sql(f"DELETE FROM {delta('events_delta')} WHERE country = 'JP' AND amount < 1")

In [ ]:
# Compare the two DELETEs in the history (look at operationMetrics)
display(spark.sql(f"DESCRIBE HISTORY {delta('events_delta')}")
        .select("version", "operation", "operationMetrics")
        .orderBy("version"))

In [ ]:
# Time travel: the old versions are still queryable
display(spark.sql(f"""
SELECT 'current' AS version, count(*) AS rows FROM {delta('events_delta')}
UNION ALL
SELECT 'version 0', count(*) FROM {delta('events_delta')} VERSION AS OF 0
"""))

In [ ]:
# [BEFORE CLASS] Export identical Parquet for the Spark vs DuckDB comparison
(spark.read.format("delta").load(f"{BASE}/events_delta")
    .repartition(16).write.mode("overwrite").parquet(f"{BASE}/export/events"))
(spark.read.format("delta").load(f"{BASE}/users_delta")
    .repartition(4).write.mode("overwrite").parquet(f"{BASE}/export/users"))

spark.read.parquet(f"{BASE}/export/events").createOrReplaceTempView("events")
spark.read.parquet(f"{BASE}/export/users").createOrReplaceTempView("users")
print("exported")

## 1.2 Execution showdown
Same join + aggregation, three ways on Spark, then DuckDB on the same machine.
40% of events belong to one user, so a hash-partitioned join puts 40% of the work on one task.

In [ ]:
Q_JOIN = """
SELECT u.tier, e.country, count(*) AS n, round(sum(e.amount), 2) AS revenue
FROM events e JOIN users u ON e.user_id = u.user_id
GROUP BY u.tier, e.country
ORDER BY revenue DESC
"""

In [ ]:
# Step 1: the naive run (no AQE, no broadcast)  -> Spark UI: the 200-task stage, Summary Metrics max vs median
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.shuffle.partitions", "200")

spark.sql(Q_JOIN).explain("formatted")   # expect SortMergeJoin with Exchange on both sides
timed("1.2 naive SMJ", Q_JOIN)

In [ ]:
# Skew, done by hand: rows each of the 200 join tasks receives
# Spark's hash partitioning is pmod(murmur3_hash(key), numPartitions), which is exactly F.hash.
# So this reproduces the row count every shuffle partition (= every join task) got in Step 1.
parts = (spark.table("events")
         .groupBy(F.pmod(F.hash("user_id"), F.lit(200)).alias("task"))
         .count().toPandas().sort_values("count", ascending=False))
med, mx = int(parts["count"].median()), int(parts["count"].max())
print(f"median task: {med:,} rows | biggest task: {mx:,} rows | {mx / med:,.0f}x the median")
display(parts.head(10))

In [ ]:
# Step 2a: fix with AQE skew-join handling (splits the hot partition)
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", str(32 * 1024 * 1024))
timed("1.2 AQE skew join", Q_JOIN)
# In the Spark UI (localhost:4040) SQL / DataFrame tab, the SortMergeJoin node now shows skew=true / split partitions

In [ ]:
# Step 2b: fix with a broadcast join (no shuffle of the big side at all)
Q_JOIN_BCAST = Q_JOIN.replace("SELECT u.tier", "SELECT /*+ BROADCAST(u) */ u.tier", 1)
spark.sql(Q_JOIN_BCAST).explain("formatted")   # expect BroadcastHashJoin, no Exchange on events
timed("1.2 broadcast join", Q_JOIN_BCAST)

In [ ]:
# Step 3: identical SQL in DuckDB, same files, same machine
import duckdb
con = duckdb.connect()
con.execute(f"PRAGMA threads={os.cpu_count()}")

Q_DUCK = Q_JOIN.replace("FROM events e", f"FROM read_parquet('{LOCAL}/export/events/*.parquet') e") \
               .replace("JOIN users u", f"JOIN read_parquet('{LOCAL}/export/users/*.parquet') u")

t0 = time.time()
duck_result = con.sql(Q_DUCK).df()
TIMINGS["1.2 DuckDB, single process"] = round(time.time() - t0, 2)
print(f"DuckDB: {TIMINGS['1.2 DuckDB, single process']}s on {os.cpu_count()} threads")
display(duck_result.head(10))

In [ ]:
# Sanity check: both engines agree
spark_total = spark.sql(Q_JOIN).agg(F.sum("n").alias("n"), F.round(F.sum("revenue"), 0).alias("rev")).collect()[0]
print("Spark :", spark_total["n"], spark_total["rev"])
print("DuckDB:", int(duck_result["n"].sum()), round(duck_result["revenue"].sum()))

In [ ]:
# Scoreboard (screenshot this for the slide)
display(pd.DataFrame(TIMINGS.items(), columns=["run", "seconds"]))

In [ ]:
# Bonus (bridge to Case Study 2): DuckDB reading the Delta table directly
# Needs the cluster to reach DuckDB's extension repo. Skip if it fails.
try:
    con.execute("INSTALL delta; LOAD delta;")
    display(con.sql(f"SELECT country, count(*) AS n FROM delta_scan('{LOCAL}/events_delta') GROUP BY 1 ORDER BY 2 DESC").df())
except Exception as e:
    print("delta extension unavailable here:", e)